# FAIRLens Cloud Generation Notebook
## Cross-Model Demographic Representational Disparity Audit

> **Scientific Protocol:**
> - Audits demographic representational disparity using counterfactual prompt pairs.
> - Evaluates ONE canonical model per run under strict 4-bit NF4 quantization.
> - Three execution modes:
>   1. **`smoke`**: 5 pairs × 2 conditions × K=3 = **30 generations** (infrastructure verification).
>   2. **`pilot`**: 20 stratified pairs × 2 conditions × K=20 = **800 generations** (cross-domain pilot audit).
>   3. **`full`**: 360 pairs × 2 conditions × K=20 = **14,400 generations/model** (canonical audit).
> - Uses the frozen FAIRLens v1 corpus ($N=360$ prompt pairs) verified by SHA-256.
> - Each sample uses a distinct deterministically derived pseudorandom seed via SHA-256.
> - Crash-safe checkpointing to SQLite after every completed sample; resumes seamlessly.

---

### Instructions for Cloud Execution

#### On Kaggle:
1. Settings -> Accelerator -> GPU (P100 or T4 x2).
2. Settings -> Secrets -> Add Secret `HF_TOKEN` with your Hugging Face user access token.
3. Set `RUN_MODE = "smoke"` first to verify the pipeline (30 generations total).
4. Next, set `RUN_MODE = "pilot"` to run the 800-sample cross-domain pilot audit.
5. Once pilot passes, set `RUN_MODE = "full"` and select your target model.

#### On Google Colab:
1. Runtime -> Change runtime type -> Hardware accelerator -> T4 or A100 GPU.
2. Secrets (key icon) -> Add `HF_TOKEN` with your Hugging Face token.
3. Run cells in order.

In [ ]:
# ---------------------------------------------------------------------------
# Cell 1: Environment Setup & Dependency Installation
# ---------------------------------------------------------------------------
import os
import sys
import subprocess
from pathlib import Path

# Check if running inside cloned repo or fresh environment
if not Path("fairlens").exists() and not Path("../fairlens").exists():
    print("Cloning repository...")
    !git clone https://github.com/ArafathUIU/Research-Bias.git
    %cd Research-Bias
elif Path("../fairlens").exists():
    %cd ..

print(f"Working directory: {Path.cwd()}")
sys.path.insert(0, str(Path.cwd()))

# Install required dependencies for 4-bit generation
!pip install -q -U transformers accelerate bitsandbytes sentence-transformers psutil
print("Dependencies installed successfully.")

In [ ]:
# ---------------------------------------------------------------------------
# Cell 2: GPU Detection & System Diagnostics
# ---------------------------------------------------------------------------
import torch
import psutil
import platform

print("=" * 60)
print("SYSTEM & GPU DIAGNOSTICS")
print("=" * 60)
print(f"Python Version   : {platform.python_version()}")
print(f"PyTorch Version  : {torch.__version__}")
print(f"CUDA Available   : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device_count = torch.cuda.device_count()
    print(f"CUDA Device Count: {device_count}")
    for i in range(device_count):
        prop = torch.cuda.get_device_properties(i)
        vram = prop.total_memory / (1024 ** 3)
        print(f"  Device {i}: {prop.name} ({vram:.2f} GB VRAM, CC {prop.major}.{prop.minor})")
else:
    print("WARNING: No CUDA device detected! Generation requires a GPU.")

ram = psutil.virtual_memory().total / (1024 ** 3)
print(f"System RAM       : {ram:.1f} GB")
print("=" * 60)

In [ ]:
# ---------------------------------------------------------------------------
# Cell 3: Hugging Face Authentication (Secrets Only — No Hard-Coding)
# ---------------------------------------------------------------------------
import os
import huggingface_hub

hf_token = os.environ.get("HF_TOKEN")

# Check Kaggle secrets
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        hf_token = user_secrets.get_secret("HF_TOKEN")
    except Exception:
        pass

# Check Colab userdata
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        pass

if hf_token:
    huggingface_hub.login(token=hf_token)
    print("Hugging Face authenticated successfully via secret.")
else:
    print("NOTE: No HF_TOKEN found in secrets. Gated models (Gemma-2, Mistral) will require a token.")

In [ ]:
# ---------------------------------------------------------------------------
# Cell 4: Experiment Configuration & Execution Mode Selection
# ---------------------------------------------------------------------------
# SELECT EXECUTION MODE:
#   "smoke" : Infrastructure test (Phi-3-mini, 5 pairs, K=3 -> 30 generations)
#   "pilot" : Stratified cross-domain audit (Phi-3-mini, 20 pairs across 10 domains, K=20 -> 800 generations)
#   "full"  : Production audit (Selected model, all 360 pairs, K=20 -> 14,400 generations)
RUN_MODE = "smoke"   # "smoke" | "pilot" | "full"

# CANONICAL MODEL SUITE (Used in "full" mode; "smoke" and "pilot" use Phi-3-mini):
#   'phi3_mini'    -> microsoft/Phi-3-mini-4k-instruct
#   'qwen25_7b'    -> Qwen/Qwen2.5-7B-Instruct
#   'mistral_7b'   -> mistralai/Mistral-7B-Instruct-v0.3
#   'gemma2_9b'    -> google/gemma-2-9b-it
#   'deepseek_r1'  -> deepseek-ai/DeepSeek-R1-Distill-Llama-8B
SELECTED_MODEL_KEY = "phi3_mini"

CANONICAL_MODELS_MAP = {
    "phi3_mini": "microsoft/Phi-3-mini-4k-instruct",
    "qwen25_7b": "Qwen/Qwen2.5-7B-Instruct",
    "mistral_7b": "mistralai/Mistral-7B-Instruct-v0.3",
    "gemma2_9b": "google/gemma-2-9b-it",
    "deepseek_r1": "deepseek-ai/DeepSeek-R1-Distill-Llama-8B",
}

# Generation Hyperparameters (fixed across all runs per protocol)
PRIMARY_TEMPERATURE = 0.7
TOP_P = 0.95
MAX_NEW_TOKENS = 128

if RUN_MODE == "smoke":
    MODEL_KEY = "phi3_mini"
    MODEL_NAME = CANONICAL_MODELS_MAP[MODEL_KEY]
    NUM_SAMPLES = 3            # K = 3 for smoke test
    OUTPUT_DIR = "results/smoke"
    EXPERIMENT_ID = f"smoke_{MODEL_KEY}_T07_K3"
    print("=" * 60)
    print("RUN MODE: SMOKE TEST (Infrastructure Validation)")
    print(f"Model          : {MODEL_NAME}")
    print(f"Pairs          : 5 pairs x 2 conditions x K={NUM_SAMPLES} = 30 generations")
    print(f"Output Dir     : {OUTPUT_DIR} (quarantined from full results)")
    print("=" * 60)
elif RUN_MODE == "pilot":
    MODEL_KEY = "phi3_mini"
    MODEL_NAME = CANONICAL_MODELS_MAP[MODEL_KEY]
    NUM_SAMPLES = 20           # K = 20 for pilot audit
    OUTPUT_DIR = "results/pilot"
    EXPERIMENT_ID = f"pilot_{MODEL_KEY}_T07_K20"
    print("=" * 60)
    print("RUN MODE: PILOT TEST (Stratified Cross-Domain Audit)")
    print(f"Model          : {MODEL_NAME}")
    print(f"Pairs          : 20 stratified pairs x 2 conditions x K={NUM_SAMPLES} = 800 generations")
    print(f"Output Dir     : {OUTPUT_DIR} (quarantined from full results)")
    print("=" * 60)
elif RUN_MODE == "full":
    MODEL_KEY = SELECTED_MODEL_KEY
    MODEL_NAME = CANONICAL_MODELS_MAP[MODEL_KEY]
    NUM_SAMPLES = 20           # K = 20 for full audit
    OUTPUT_DIR = "results/generations"
    EXPERIMENT_ID = f"fairlens_v1_{MODEL_KEY}_T07_K20_ind"
    print("=" * 60)
    print("RUN MODE: FULL FAIRLENS V1 AUDIT")
    print(f"Model          : {MODEL_NAME}")
    print(f"Pairs          : 360 pairs x 2 conditions x K={NUM_SAMPLES} = 14,400 generations")
    print(f"Output Dir     : {OUTPUT_DIR}")
    print("=" * 60)
else:
    raise ValueError(f"Invalid RUN_MODE: {RUN_MODE!r}. Choose 'smoke', 'pilot', or 'full'.")

In [ ]:
# ---------------------------------------------------------------------------
# Cell 5: Frozen Corpus Loading & Verification
# ---------------------------------------------------------------------------
import hashlib
import json
from pathlib import Path
from fairlens.corpus import select_stratified_pilot_pairs

corpus_path = Path("results/corpus/audit_corpus_fairlens_v1.jsonl")
manifest_path = Path("results/corpus/corpus_manifest_fairlens_v1.json")

assert corpus_path.exists(), f"Corpus file not found: {corpus_path}"
assert manifest_path.exists(), f"Corpus manifest not found: {manifest_path}"

# Verify SHA-256 of corpus file
with open(manifest_path, "r", encoding="utf-8") as f:
    corpus_manifest = json.load(f)

expected_sha = corpus_manifest["corpus_file_sha256"]

hasher = hashlib.sha256()
with open(corpus_path, "rb") as f:
    for chunk in iter(lambda: f.read(65536), b""):
        hasher.update(chunk)
actual_sha = hasher.hexdigest()

print(f"Corpus Expected SHA-256: {expected_sha}")
print(f"Corpus Actual   SHA-256: {actual_sha}")
assert actual_sha == expected_sha, "CRITICAL: Corpus SHA-256 mismatch! Frozen corpus has drifted."
print("Corpus integrity verified successfully!")

# Load pairs
all_pairs = []
with open(corpus_path, "r", encoding="utf-8") as f:
    for line in f:
        all_pairs.append(json.loads(line))

print(f"Total pairs in frozen corpus: {len(all_pairs)}")

# Select pairs according to execution mode
if RUN_MODE == "smoke":
    active_pairs = all_pairs[:5]
    print(f"Smoke mode: using first {len(active_pairs)} pairs.")
elif RUN_MODE == "pilot":
    active_pairs = select_stratified_pilot_pairs(all_pairs)
    print(f"Pilot mode: selected {len(active_pairs)} stratified pairs (10 main + 10 SES across all 10 domains).")
else:
    active_pairs = all_pairs
    print(f"Full mode: using all {len(active_pairs)} pairs.")

In [ ]:
# ---------------------------------------------------------------------------
# Cell 6: Model & Tokenizer Loading (4-Bit NF4 Quantization)
# ---------------------------------------------------------------------------
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print(f"Loading model: {MODEL_NAME} ...")

# 4-bit NF4 Quantization Configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

# Set padding token if missing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

# Inspect model revision / git commit if available
model_revision = getattr(model.config, "_commit_hash", "unknown")
print(f"Model loaded successfully!")
print(f"Revision Hash : {model_revision}")
print(f"Device Map    : {model.hf_device_map if hasattr(model, 'hf_device_map') else 'auto'}")

In [ ]:
# ---------------------------------------------------------------------------
# Cell 7: Checkpoint Store & Resume Initialization
# ---------------------------------------------------------------------------
from fairlens.storage import GenerationStore
from fairlens.seeds import derive_sample_seed

out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

db_path = out_dir / f"{EXPERIMENT_ID}.db"
jsonl_path = out_dir / f"{EXPERIMENT_ID}.jsonl"

store = GenerationStore(str(db_path))
store.create_tables()

completed_keys = store.get_completed_keys(EXPERIMENT_ID)
total_target = len(active_pairs) * 2 * NUM_SAMPLES
print(f"Checkpoint DB       : {db_path}")
print(f"Already completed   : {len(completed_keys)} / {total_target} samples")
print(f"Remaining to generate: {total_target - len(completed_keys)} samples")

In [ ]:
# ---------------------------------------------------------------------------
# Cell 8: Resumable Generation Loop with Deterministic Per-Sample Seeds
# ---------------------------------------------------------------------------
import time
from datetime import datetime, timezone

print(f"Starting generation loop ({RUN_MODE.upper()} MODE)...")
start_time = time.time()
generated_count = 0
skipped_count = 0
refusal_count = 0

REFUSAL_KEYWORDS = [
    "i cannot", "i am unable", "as an ai", "i'm sorry, but",
    "against my ethical guidelines", "content policy",
]

for pair_idx, pair in enumerate(active_pairs):
    pair_id = pair["pair_id"]
    domain = pair["domain"]
    group_a = pair["group_a"]
    group_b = pair["group_b"]
    template_idx = pair["template_idx"]

    for condition in ["A", "B"]:
        prompt = pair["prompt_a"] if condition == "A" else pair["prompt_b"]

        for sample_idx in range(NUM_SAMPLES):
            key = (EXPERIMENT_ID, pair_id, condition, sample_idx)
            if key in completed_keys:
                skipped_count += 1
                continue

            # Derive deterministic per-sample seed via SHA-256
            sample_seed = derive_sample_seed(
                experiment_id=EXPERIMENT_ID,
                model_id=MODEL_KEY,
                pair_id=pair_id,
                condition=condition,
                sample_idx=sample_idx,
            )
            torch.manual_seed(sample_seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(sample_seed)

            # Format input using chat template where available
            messages = [{"role": "user", "content": prompt}]
            if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template is not None:
                input_text = tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True,
                )
            else:
                input_text = prompt

            inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
            prompt_tokens_len = inputs["input_ids"].shape[-1]

            t0 = time.time()
            with torch.no_grad():
                output_ids = model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    temperature=PRIMARY_TEMPERATURE,
                    top_p=TOP_P,
                    do_sample=True,
                    pad_token_id=tokenizer.eos_token_id,
                )
            gen_time_ms = (time.time() - t0) * 1000.0

            # Slice off input tokens to isolate the generated response
            new_tokens = output_ids[0][prompt_tokens_len:]
            response_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

            # Check for empty response or refusal
            is_empty = len(response_text) == 0
            is_refusal = any(kw in response_text.lower() for kw in REFUSAL_KEYWORDS)
            if is_refusal:
                refusal_count += 1

            record = {
                "experiment_id": EXPERIMENT_ID,
                "model_id": MODEL_KEY,
                "model_revision": str(model_revision),
                "pair_id": pair_id,
                "group": condition,
                "sample_idx": sample_idx,
                "domain": domain,
                "group_a": group_a,
                "group_b": group_b,
                "template_idx": template_idx,
                "prompt": prompt,
                "response": response_text,
                "temperature": PRIMARY_TEMPERATURE,
                "top_p": TOP_P,
                "max_new_tokens": MAX_NEW_TOKENS,
                "precision": "float16",
                "quantization": "nf4_4bit",
                "device": str(model.device),
                "seed": sample_seed,
                "runtime_ms": round(gen_time_ms, 2),
                "timestamp": datetime.now(timezone.utc).isoformat(),
            }

            # Atomic checkpoint insert into SQLite
            store.insert_sample(record)
            completed_keys.add(key)
            generated_count += 1

            # Periodic logging
            log_interval = 10 if RUN_MODE in ("smoke", "pilot") else 50
            if generated_count % log_interval == 0 or generated_count == total_target:
                elapsed = time.time() - start_time
                rate = generated_count / elapsed if elapsed > 0 else 0
                print(
                    f"Progress: {len(completed_keys)}/{total_target} samples "
                    f"({generated_count} generated, {skipped_count} skipped, "
                    f"{refusal_count} refusals) [{rate:.2f} samples/s]"
                )

print("=" * 60)
print("GENERATION COMPLETE!")
print(f"Total completed in DB : {len(store.get_completed_keys(EXPERIMENT_ID))}")
print(f"Total refusals logged : {refusal_count}")
print("=" * 60)

In [ ]:
# ---------------------------------------------------------------------------
# Cell 9: Export to JSONL, Manifest Generation & GPU Cleanup
# ---------------------------------------------------------------------------
import gc
from fairlens.manifest import make_manifest

# 1. Export SQLite samples to JSONL
exported_count = store.export_jsonl(str(jsonl_path))
print(f"Exported {exported_count} samples to JSONL: {jsonl_path}")

# 2. Build and save experiment manifest (recording selected_pair_ids for pilot/smoke)
selected_ids = [p["pair_id"] for p in active_pairs] if RUN_MODE in ("smoke", "pilot") else None

manifest = make_manifest(
    model_name=MODEL_NAME,
    model_id=MODEL_KEY,
    experiment_track=f"fairlens_v1_{RUN_MODE}" if RUN_MODE != "full" else "fairlens_v1",
    primary_temperature=PRIMARY_TEMPERATURE,
    top_p=TOP_P,
    num_samples=NUM_SAMPLES,
    max_new_tokens=MAX_NEW_TOKENS,
    model_revision=str(model_revision),
    corpus_pair_count=len(active_pairs),
    corpus_manifest_sha256=expected_sha,
    selected_pair_ids=selected_ids,
    output_dir=str(out_dir),
)
manifest_saved_path = manifest.save(str(out_dir))
print(f"Experiment manifest saved: {manifest_saved_path}")
print(f"Manifest SHA-256          : {manifest.sha256}")

# 3. Free GPU memory
del model
del tokenizer
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()
print("GPU memory cleaned successfully.")